# Stable Diffusion

# VAE 模型架构深度解析:
## 1. 核心设计思想
- **变分自编码器 (VAE)** 是一种无监督学习模型，用于将高维数据压缩到低维空间，并生成新的数据样本。
- 核心是通过 “概率建模” 学习输入数据的潜在分布（Latent Distribution），既能实现数据重建，又能通过潜在分布生成新数据。与传统自编码器（AE）相比，VAE 的潜在空间是连续且结构化的，支持通过随机采样生成全新样本（如生成新图像、文本）。
- 在Stable Diffusion中承担图像⇄潜在空间的转换：

- 流程：输入数据 → 编码器（输出分布参数）→ 潜在分布采样 → 解码器（重建输入）
- **编码器**：将图像压缩(映射)到4*64×64的潜在空间,输出分布的参数（均值和方差）
- **采样**：从潜在空间分布中使用重参数化技巧，进行采样，生成新的潜空间向量
- **解码器**：将采样潜在向量并重建为3*512×512的RGB图像

## 2、VAE 编码器结构（AutoencoderKL.encoder）
- **流程**：输入数据 → 编码器（输出分布参数）→ 潜在分布采样 → 解码器（重建输入）
- **编码器**：将图像压缩(映射)到4*64×64的潜在空间,输出分布的参数（均值和方差）
- **采样**：从潜在空间分布中使用重参数化技巧，进行采样，生成新的潜空间向量
- **解码器**：将采样潜在向量并重建为3*512×512的RGB图像

### 1. 输入层
- 卷积层 conv_in:
    - 输入：RGB 图像（3 通道）
    - 作用：将原始图像投影到特征空间，通道数扩展为 128
    - 实现：Conv2d(3, 128, kernel_size=3, stride=1, padding=1)
### 2. 下采样模块 (down_blocks)
down_blocks 是一个 ModuleList，包含多个 DownEncoderBlock2D。每个模块内部有：
- ResnetBlock2D：多个残差块，用于增强非线性建模能力
- Downsample2D：下采样操作（通常通过 stride=2 的卷积实现）
#### 2.1 DownEncoderBlock2D (0)
- 通道数保持：128 → 128
- 包含：2 个 ResNet block + 1 个 Downsample2D
- 作用：下采样使空间尺寸缩小一半，分辨率从512*512-》256*256
#### 2.2 DownEncoderBlock2D (1)
- 通道数变换：128 → 256
- 使用：conv_shortcut 实现残差维度匹配
- 下采样：Conv2d(256, 256, stride=2)
- 作用：继续下采样 分辨率为128*128
#### 2.3 DownEncoderBlock2D (2)
- 通道数变换：256 → 512
- 使用：conv_shortcut 实现维度扩展
- 下采样：Conv2d(512, 512, stride=2)
- 作用：继续下采样 分辨率为64*64
#### 2.4 DownEncoderBlock2D (3)
- 通道数保持：512
- 作用：编码器的最底层，包含两个 ResNet block 提取高层语义信息
### 3. 中间模块 (mid_block)
- 包含：
    - 两个 ResnetBlock2D
    - 一个 Attention 模块（基于 self-attention，使用 QKV + GroupNorm）
- 作用：在最小分辨率下对特征进行非线性变换，增强模型对全局上下文的理解
### 4. 输出部分 (conv_out)
- 卷积层 conv_out：将通道数进行压缩，以减少特征图的维度，并生成最终的生成结果。
    - 输入：512 通道
    - 输出：8 通道（潜在变量的均值和 log-variance）
    - 实现：Conv2d(512, 8, kernel_size=3, stride=1, padding=1)
### 5. VAE 损失函数
- **VAE的损失函数由三部分组成**：重建损失（Reconstruction Loss） ，LPIPS 感知损失（LPIPS Loss）和KL 散度（KL Divergence），整体损失为三者之和：$L=L_{recon}+L_{LPIPS}+L_{KL}$

- **重建损失（$L_recon$​）**
衡量 “重建数据” 与 “原始输入” 的差异，确保解码器能从潜在向量中还原输入。
    对于图像（像素值∈[0,1]）：常用二元交叉熵（BCE）
    对于连续值数据（如自然图像）：可用均方误差（MSE）


- **LPIPS 感知损失（$L_{LPIPS}$​）**
衡量 “重建数据” 与 “原始输入” 的差异，确保解码器能从潜在向量中还原输入。
    对于图像（像素值∈[0,1]）：常用二元交叉熵（BCE）
    对于连续值数据（如自然图像）：可用均方误差（MSE）

- **KL 散度（$L_{KL}$​）**
衡量 “编码器输出的潜在分布N(μ,σ2)” 与 “预设的先验分布N(0,1)” 的差异，起到正则化作用。

    作用：强制潜在分布接近标准正态分布，使潜在空间连续且结构化（任意采样的 z 都能生成有意义的样本）。
- 以 MNIST 为例（像素值归一化到 [0,1]），BCE 损失公式：
$$
\mathcal{L}_{\text{recon}} = -\sum_{i=1}^{N} \left[ x_i \log(x_i^{\text{recon}}) + (1 - x_i) \log(1 - x_i^{\text{recon}}) \right]
$$

- 其中$x_i$​是原始像素，$x^i_{\text{recon}}$​是重建像素。


- LPIPS 感知损失公式：
$$
LPIPS(x,y)=∑​w_l​⋅d_l​(x,y)
$$


- KL 散度的解析解：
$$
\mathcal{L}_{\text{KL}} = -\frac{1}{2} \sum_{i=1}^{Z} \left( 1 + \log \sigma_i^2 - \mu_i^2 - \sigma_i^2 \right)
$$

- 其中 Z 是潜在空间维度，$μ_i$​和$σ_i$​是第 i 个维度的均值和标准差。

## 三、各模块功能解析
| 模块名称 | 功能简介 |
|---------|---------|
| ResnetBlock2D | 提升模型非线性建模能力，包含归一化、激活、残差连接 |
| GroupNorm | 特征通道归一化，稳定训练 |
| SiLU 激活 | 平滑非线性激活函数 |
| conv_shortcut | 1x1 卷积匹配残差维度 |
| Attention 模块 | 提升网络对远距离特征的建模能力 |
| Conv2d+Stride/2 | 用于下采样 |
| Upsample2D | 用于上采样 |



## 四、VAE 输出的 8 通道到 UNet 输入的 4 通道转换
### 1. VAE 编码器输出 8 通道
- 前 4 个通道：均值 μ
- 后 4 个通道：log 方差 log(σ²)
### 2. Reparameterization Trick
- 步骤：
    - VAE encoder 输出 8 通道的张量 → (B, 8, H, W)
    - 拆分成两半：mean 和 logvar → (B, 4, H, W) + (B, 4, H, W)
    - 重参数采样（推理时可以直接取 mean）：
        - std = torch.exp(0.5 * logvar)
        - noise = torch.randn_like(std)
        - z = mean + std * noise
    - 最终 z 是 4 通道的 latent → (B, 4, H, W)，用于传给 UNet。

## 推断过程（Sampling）

从纯噪声逐步逆扩散得到图像，公式：
$$
z_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( z_t - \frac{\beta_t}{\sqrt{1 - \alpha_t}} \, \epsilon_\theta(z_t, t, c) \right) + \sigma_t \cdot \eta
$$

流程如下：

- 初始化随机噪声向量 $z_T$ ~ N(0, I)

- 迭代从 t = T 到 t = 1：

   - 根据预测的 $ε_θ(z_t, t, c) 计算 z_{t-1}$

   - 最后一步得到 $z_0$

- VAE Decoder 将 $z_0 解码为图像 x$

## VAE 优缺点与改进
- **优点**：

    可生成性：潜在空间连续，支持随机采样生成新样本（传统 AE 的潜在空间是离散的，无法生成）。
    理论严谨：基于变分推断（Variational Inference），有明确的概率解释。
    灵活性：适用于图像、文本、音频等多种数据类型。

- **缺点**：

    重建质量有限：KL 散度的正则化可能导致重建图像模糊（如 MNIST 生成的数字边缘不清晰）。
    潜在空间控制弱：难以通过潜在向量的特定维度控制生成样本的属性（如控制数字 “粗细”）。

- **改进方向**：

    β-VAE：在 KL 散度前加系数 β（β>1），增强潜在空间的结构化（代价是重建质量下降）。
    CVAE（Conditional VAE）：加入条件信息（如类别标签），生成指定类别的样本。
    VQ-VAE：引入离散潜在空间（向量量化），提升重建质量和生成可控性。


## 总结
VAE 通过 “概率潜在分布” 和 “重参数化技巧”，实现了从 “重建” 到 “生成” 的跨越，是生成模型的基础架构之一。其核心是损失函数中 “重建损失” 与 “KL 散度” 的平衡 —— 前者保证重建准确性，后者保证潜在空间的可生成性。理解 VAE 的架构和损失设计，是掌握更复杂生成模型（如 GAN、Diffusion Models）的基础